# Notebook 2: Math Refresher, Fundamental Functions and Calculus

*Quantitative Methods in Neuroscience, Master in Neuroscience, University of Geneva (2026-27)*

This is the practical companion to **Lecture 2, Calculus Refresher and Fundamental Functions
for Neuroscience**. The lecture introduced a small zoo of mathematical objects that come back
again and again in experimental neuroscience: exponentials, logarithms, Gaussians, sigmoids,
sines and cosines, together with the calculus used to manipulate them, derivatives and
integrals.

Here we build those functions ourselves, plot them, take them apart parameter by parameter,
and then do calculus on them **numerically**, on a computer, rather than with pen and paper.
Everything in this notebook runs on signals we generate ourselves, so there is no dataset to
load: real data arrives next week.

### By the end of this notebook you will be able to:
- Write your own Python **functions**, with arguments, defaults and a docstring.
- Evaluate a function on a grid of points with **NumPy arrays**, without writing a loop.
- Build a figure deliberately: **figure and axes**, lines, points, bars, error bars, labels,
  legends, several panels, and saving the result to a file.
- Recognise the **fundamental shapes** and say what each of their parameters does.
- Compute a **numerical derivative** and a **numerical integral**, and check them against the
  analytic answer.
- State the **Fundamental Theorem of Calculus** and verify it numerically.
- Approximate a function near a point with its **first-order Taylor expansion**, and see where
  that approximation breaks down.
- Put all of it together on generated signals: measure a time constant from a noisy trace,
  and find the moments a learning curve changes.

### Table of contents
0. [Setting up](#section-0)
1. [Functions and how to plot them](#section-1)
   - 1.1 [Defining a function](#section-1-1)
   - 1.2 [Reusing your code: a module](#section-1-2) 🔬
   - 1.3 [NumPy arrays and vectorization](#section-1-3)
   - 1.4 [Anatomy of a plot](#section-1-4)
   - 1.5 [The function zoo](#section-1-5)
2. [Calculus, numerically](#section-2)
   - 2.1 [The derivative as a local slope](#section-2-1)
   - 2.2 [The second derivative: curvature](#section-2-2)
   - 2.3 [The integral as accumulation](#section-2-3)
   - 2.4 [The Fundamental Theorem of Calculus](#section-2-4) 🔬
   - 2.5 [Local linearization: Taylor to first order](#section-2-5) 🔬
3. [Putting it together on generated signals](#section-3) 🔬
   - 3.1 [A calcium transient](#section-3-1)
   - 3.2 [When did the animal learn?](#section-3-2)
4. [What comes next](#section-4)

> **How to read this notebook.** Markdown cells like this one explain *why* we are doing
> something and the *concept* behind the maths. Code cells implement it. Look out for
> **`▶ Task`** boxes: those are short exercises where you edit or extend the code. Each task
> carries a difficulty tag:
>
> - **⭐** essential for everyone: run the code, understand it, change it slightly.
> - **⭐⭐** intermediate: repair a small bug, complete the missing lines, write a short function.
> - **⭐⭐⭐** optional challenge: build a fuller piece of analysis on your own.
>
> Some section titles carry a **🔬**. Those sections go further than the course strictly
> requires, and you can skip them lightheartedly: nothing that comes afterwards will stop making
> sense. Each one says in a line why it is there and whether the material comes back later. Tasks
> inside a 🔬 section are optional too, whatever their stars.
>
> If you get stuck, ask the TAs. If you use AI help, treat it as a tutor and not as a
> replacement for thinking: you should still be able to read every line and explain what it
> does and why. We suggest the custom bot prepared for this course:
> https://chatgpt.com/g/g-69fc9bfc9a748191a912a124800d9254-qmn-teaching-assistant-bot (to access sign in with a free OpenAI account)

<a id="section-0"></a>
## 0. Setting up

> **Before you run anything**, make sure the course conda environment **`qmn`** is active, or
> selected as this notebook's kernel in VS Code. If you have not created it yet, follow the
> illustrated setup guide (`QMN_setup_guide_students.pdf`) or `SETUP_INSTRUCTIONS.txt`, both in
> the course folder.

In [ ]:
# Where the data and the shared helpers live. This works whether the notebook
# is opened from notebooks/ or from a subfolder of it.
import sys
from pathlib import Path

# sys.path is the list of folders Python searches when you import something. Adding ROOT to it
# is what lets `from src.qmn_utils import ...` work further down, whatever folder you started in.

ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
# --- Import the scientific Python stack
import numpy as np                  # import of Numpy library: numerical arrays and maths
import matplotlib.pyplot as plt     # import of Matplotlib library: plotting
from cycler import cycler           # to set our own default colour sequence

# A bit of style for every figure in this notebook (feel free to tweak it)
plt.rcParams.update({
    "figure.figsize": (6, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.prop_cycle": cycler(color=[
        "#1f77b4",  # blue
        "#ff7f0e",  # orange
        "#5b2c83",  # purple
    ]),
})

print(f"NumPy      : {np.__version__}")
print(f"matplotlib : {plt.matplotlib.__version__}")

<a id="section-1"></a>
# Part 1. Functions and how to plot them

Three things, in the order in which each one needs the one before it: how to write a
**function**, how to hold a lot of numbers at once in an **array**, and how to **plot** the
result. By the end of this part you will have defined a function, evaluated it on a grid of
points, and drawn it.

<a id="section-1-1"></a>
## 1.1 Defining a function

A **function** is a reusable block of code: you give it inputs (its *arguments*) and it hands
back a result with `return`. Think of it as a small machine. Functions are the main tool we
have for not repeating ourselves: if you catch yourself copy-pasting a calculation, that is the
signal to turn it into a function. Then a change of mind means editing one place instead of
five.

This is the first thing you write yourself that Week 1 did not cover, so here is the shape of it,
line by line:

```python
def function_name(first_input, second_input):     # the keyword def, a name, the inputs, a colon
    """One sentence saying what the function does."""
    result = first_input + second_input           # the body, indented, exactly like an if or a for
    return result                                 # hand the value back to whoever called it
```

Two things worth saying out loud. The names in the brackets are **placeholders**: they stand for
whatever the caller passes in, and they only exist inside the function. And **`return` is not
`print`**: printing shows you something and gives back nothing, while returning hands you a value
you can store in a variable and use later. Nine times out of ten you want `return`.

The `"""` line just under `def` is the **docstring**: one sentence saying what the function
does. Write one every time. In six weeks it will be you reading it.

In [ ]:
# --- A very first function
def square(x):
    """Return x squared."""
    return x * x

print(square(3))       # 9
print(square(0.5))     # 0.25

Arguments can have **default values**, used whenever the caller does not supply them. Here is the
**logistic sigmoid**, the S shaped curve we will meet again in Week 3 as a psychometric curve.

$$\sigma(x; \, k, x_0) \;=\; \frac{1}{1 + e^{-k\,(x - x_0)}}$$

with $k$ controlling how steep the S is, and $x_0$ shifting it left or right.

One thing in the code below is borrowed from the next section: `np.exp` is NumPy's exponential function, $e^x$. For now read it as "e to the power of", and section 1.3 explains where `np` comes from and why we use its version rather than the one in the standard library.

In [ ]:
# --- A logistic sigmoid, with defaults for slope and bias
def sigmoid(x, slope=1.0, bias=0.0):
    """Logistic sigmoid with adjustable slope and horizontal bias."""
    return 1.0 / (1.0 + np.exp(-slope * (x - bias)))

print(sigmoid(0.0))                  # 0.5 by construction
print(sigmoid(10.0))                 # almost 1
print(sigmoid(10.0, slope=0.1))      # much less, the curve is flatter

<a id="section-1-2"></a>
## 1.2 Reusing your code: a module 🔬

> 🔬 **Optional.** Standard practice in a real project, and worth seeing once, but nothing later in
> the course imports this module.

Right now `sigmoid` lives inside this notebook. That is fine for one afternoon, but as soon as you want the
same function in another notebook, copy-pasting the definition is the easiest way to end up with two
versions that quietly disagree.

The standard practice in every real Python project is to put reusable helpers in a `.py` file,
a **module**, and import it just like NumPy. Those are the words from Week 1 section 4: a module is
one file of Python code, a library is a collection of modules installed together. Here we write our
own module instead of importing somebody else's. We keep ours in `src/qmn_utils.py`, next to the
notebooks. The cell below prints what is currently in it.

In [ ]:
# --- What is inside src/qmn_utils.py

print((ROOT / "src" / "qmn_utils.py").read_text())

In [ ]:
# --- Import from our own module, and check it agrees with the version defined above
from src.qmn_utils import sigmoid as sigmoid_from_module

# `assert` checks that a condition is true and does nothing if it is. If it is false the cell
# stops with an AssertionError and prints the message after the comma. It is how you make a
# check loud instead of hoping somebody notices a wrong number further down.
for value in [-2.0, 0.0, 2.0]:
    assert abs(sigmoid(value) - sigmoid_from_module(value)) < 1e-12, f"differ at x={value}"
print("the notebook sigmoid and the module sigmoid agree")

<a id="section-1-3"></a>
## 1.3 NumPy arrays and vectorization

In Week 1 you met the **list**, Python's general-purpose container. A list will happily hold
numbers, but it does not do arithmetic with them: multiplying a list by 2 gives you a longer
list, not bigger numbers. To do maths on every element you have to write a loop or a list
comprehension, which is fine for four numbers and painful for four hundred.

[NumPy](https://numpy.org/) is the library that fixes this, and it is the foundation of essentially all scientific
Python. Its central object is the **array**: a grid of numbers, all of the same type, that
behaves like a number when you do arithmetic with it. We imported it at the top of the notebook under the alias `np`:

```python
import numpy as np
```

The cell below shows the difference in four lines.

In [ ]:
# --- Why an array rather than a list?
numbers_list = [1.0, 2.0, 3.0, 4.0]
numbers_array = np.array(numbers_list)      # np function to convert a list into an array

print("the list  :", numbers_list)
print("the array :", numbers_array)

# Multiplying a LIST by 2 repeats it: that is Python's definition, not a mistake.
print("\nlist  * 2 :", numbers_list * 2)

# To double each element of a list you need a comprehension
print("with a comprehension:", [value * 2 for value in numbers_list])

# Multiplying an ARRAY by 2 doubles every element, with no loop at all.
print("\narray * 2 :", numbers_array * 2)
print("array + array :", numbers_array + numbers_array)
# np function to compute the exponential of each element of an array
print("np.exp(array) :", np.exp(numbers_array).round(3))

That last line is the important one. `np.exp` applied to an array returns an array, with the
function applied to every element. Doing maths on a whole array at once, with no loop, is called
**vectorization**, and it is both faster and easier to read than the loop it replaces.

Now let us build the grid of x values we need in order to plot a function. We will do it the
long way first, with the comprehension you already know, and then convert the result into an
array.

One more thing arrays do, which we will use in a moment. Comparing an array with a number gives you
an array of `True` and `False`, one per element, called a **mask**. Put that mask in square brackets
and you get back only the elements where it was `True`.


In [ ]:
# --- Picking out the elements that satisfy a condition
values = np.array([0.4, 1.2, 0.7, 2.5, 0.9])

big = values > 1.0                  # one True or False per element: a mask
print("the mask     :", big)
print("the big ones :", values[big])   # keep only the elements where the mask is True
print("how many     :", big.sum())     # True counts as 1, as in Week 1

In [ ]:
# --- Define the grid as a Python list first
n_points = 201
x_min = -1.0
x_max = 1.0
step = (x_max - x_min) / (n_points - 1)
x_grid_list = [x_min + i * step for i in range(n_points)]

# --- Convert the Python list into a NumPy array
x_grid = np.array(x_grid_list)

# --- Evaluate the sigmoid on the whole NumPy array in one shot
y_grid = sigmoid(x_grid, slope=8.0)

print("x_grid type  :", type(x_grid)) # get info on the type of object
print("x_grid shape :", x_grid.shape) # get info on the shape of the numpy array
print("y_grid shape :", y_grid.shape)
# "slice" the array to get every 50th point, and round to 2 decimal places
print("x, every 50th point :", x_grid[::50].round(2))
print("sigmoid there       :", y_grid[::50].round(3))

Two things to notice in that output.

**`.shape`** tells you the size of an array. Here it is `(201,)`, meaning one dimension of 201
numbers. Shapes become important later, when arrays have rows and columns.

**Slicing works exactly as it does for lists.** `x_grid[::50]` means "every 50th element", which
is why the printout shows five points spread across the range instead of 201 numbers. When slicing an array the three numbers are start, stop and step. Here the start and the stop are
left empty, which means the whole array, and the step is 50.

And now the shortcut. Building a grid by hand is a useful thing to do once, so that you know what
is in it, but NumPy has a function for precisely this job: `np.linspace(start, stop, n)` returns
`n` evenly spaced points from `start` to `stop`, inclusive. From here on we will always use it.

In [ ]:
# --- The same grid, in one line
x_grid_quick = np.linspace(-1.0, 1.0, 201)

print("first three values :", x_grid_quick[:3])
print("last three values  :", x_grid_quick[-3:])
print("identical to the hand-built grid?", np.allclose(x_grid_quick, x_grid))

# The sigmoid we wrote in 1.1 is built only from NumPy operations, so it is vectorized for free:
# calling it on an array of 201 values returns an array of 201 values, no loop required.
print("\nsigmoid applied to the whole grid gives", sigmoid(x_grid_quick).shape, "values")

<a id="section-1-4"></a>
## 1.4 Anatomy of a plot

Humans are very visual animals and "a picture is worth a thousand words". Almost everything in 
this course gets checked by looking at a picture rather than raw numbers, so it is worth spending 
a little time on the vocabulary before we draw anything. This is also how we will visualise functions
like the sigmoid we defined above.

For plotting we use [Matplotlib](https://matplotlib.org/), the standard Python library for
figures, imported at the top of this notebook under the alias `plt`:

```python
import matplotlib.pyplot as plt
```

Matplotlib works with **objects**, in the sense of Week 1 section 7.2: things that carry their own
data (attributes) and their own commands (methods, called with a dot and parentheses). You will
write `ax.set_xlabel(...)` for the same reason you wrote `my_list.append(...)`. There are two such
objects you will handle all the time:

- the **figure** is the whole canvas, the thing that gets saved to a file;
- the **axes** is one set of x and y coordinates drawn inside it, with its own labels, its own
  title and its own data (typically represented by the vertical and horizontal "rulers" on the bottom 
  and left sides of a typical figure panel). A figure can hold several "frames" with different axes,
  which is what people mean by "panels" or "subplots".


One confusing detail: the matplotlib object called "axes" is a single object, not the plural
of "axis". The x axis and the y axis are parts *of* an axes.

The figure below is from the matplotlib documentation and it is the single most useful reference
you will meet this week: it names every element of a plot and, next to each name, the method that
creates it. Keep it open while you work.

<img src="assets/02/matplotlib_anatomy.png" width="620">

*Anatomy of a figure, from the [matplotlib documentation](https://matplotlib.org/stable/gallery/showcase/anatomy.html), reproduced under the matplotlib licence.*

### Your first plot

Every figure in this course starts with the same line: `plt.subplots()` creates a figure and one
axes, and hands **both** back to you at once (as a tuple). Two names on the left of the `=`, separated by a
comma, is the tuple unpacking from Week 1 section 7.4: the function returns a pair, and the two
names take one element each. After that you talk to `ax` when you want to change what is inside
the panel, and to `fig` when you want to change the whole canvas.

`plt.subplots` takes a good number of optional arguments, and its
[documentation page](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html) is
worth opening once now, if only to see what the reference page of a function you use every day
looks like.

Three lines is the minimum:

```python
fig, ax = plt.subplots()    # make a figure with one axes
ax.plot(x, y)               # draw a line on that axes
plt.show()                  # display it
```

Let us run exactly that.

In [ ]:
# --- The smallest possible plot: a line
x = np.linspace(0, 10, 100)     # 100 evenly spaced numbers from 0 to 10
y = np.sin(x)                   # the sine of each of them

fig, ax = plt.subplots()        # a figure containing one axes
ax.plot(x, y)                   # draw y against x, joined up as a line
plt.show()                      # show the result

### A second kind: the scatter

`ax.plot` joins the points with a line, which is what you want for something measured on a fine
grid, like a signal over time. When each point is a separate observation, you usually want a
**scatter**: one marker per data point and no line between them. The method is `ax.scatter`, and
everything else stays the same.

In [ ]:
# --- The same idea, one marker per observation
rng = np.random.default_rng(0)          # a random-number generator, more on this in Week 3
noise = rng.normal(0, 0.15, size=x.size)   # a little random jitter, one value per point

fig, ax = plt.subplots() # generate a figure and axes object for plotting
ax.scatter(x, y + noise) # markers instead of a line
plt.show()

### A third kind: the histogram

The two above show each observation. A **histogram** throws that detail away and shows how the
values are *distributed*: it chops the range into bins and counts how many values fall in each.
It is the workhorse of Week 3.

In [ ]:
# --- How are the values distributed?
values = rng.normal(0, 1, size=500)     # 500 random numbers

fig, ax = plt.subplots()
ax.hist(values, bins=25)                # 25 bins across the range
plt.show()

### Making a plot readable

A plot without labels is not a result, it is a doodle. The minimum any figure of yours should
carry is an **x label** and a **y label** with units, a **title** or a caption, and a **legend**
whenever more than one thing is drawn on the same axes.

The cell below takes the very first line plot and adds one thing at a time. Read it top to
bottom and match each line against the anatomy figure above.

In [ ]:
# --- The same line, dressed one element at a time
fig, ax = plt.subplots()

# colour, line width, and a name for the legend
ax.plot(x, y, color="#1f77b4", lw=2, label="sin(x)")
ax.set_xlabel("x")                                     # what the horizontal axis means
ax.set_ylabel("sin(x)")                                # what the vertical axis means
ax.set_title("A sine wave")                            # what the panel shows
ax.legend()                                            # show the labels of everything drawn
ax.axhline(0, color="gray", lw=0.8)                    # a horizontal reference line at y = 0

plt.show()

The cell below draws a decaying oscillation. Run it, then read the task under it.

In [ ]:
t_signal = np.linspace(0, 4, 400)                  # four seconds, 400 samples
signal = np.exp(-t_signal / 1.5) * np.sin(2 * np.pi * 2.0 * t_signal)

t_axis = np.linspace(0, 10, 400)                   # the grid used for the x axis

fig, ax = plt.subplots()
ax.plot(t_axis, signal, lw=1.5)
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude")
ax.set_title("a 2 Hz oscillation decaying with tau = 1.5 s")
fig.tight_layout()
plt.show()

> **▶ Task 1.1 ⭐⭐ - Repair the figure.**
> The figure above runs without an error and looks entirely plausible, and it is wrong: the picture
> does not show what the labels claim.
>
> Compare the x axis with the data being plotted, write a corrected version below, and say in a
> comment what gave the bug away.

In [ ]:
# TODO 1.1 - your corrected figure here

### Several panels in one figure

So far every figure has had exactly one axes. Asking `plt.subplots` for a grid gives you several,
and then `axes` is an array you index like any other: `axes[0, 1]` is the top-right panel.

That is the only new idea in the cell below. It draws the four plot types you will use most, one
per panel, so you can see them side by side.

In [ ]:
# --- Four plot types, one per panel
# First the data, so that the plotting code below stays easy to read.
noisy_y = y + noise                                 # from the scatter example above
groups = ["control", "drug A", "drug B"]            # three experimental conditions
means = [0.42, 0.63, 0.55]                          # their average outcome
errors = [0.05, 0.07, 0.06]                         # and the uncertainty on each average

fig, axes = plt.subplots(2, 2, figsize=(9, 6))      # a 2x2 grid: axes is now an array

axes[0, 0].plot(x, y, lw=2) # lw is the line width
axes[0, 0].set_title("plot: a line, for a continuous signal")

axes[0, 1].scatter(x, noisy_y, s=18) # s is the marker size
axes[0, 1].set_title("scatter: one marker per observation")

axes[1, 0].bar(groups, means, color="#1f77b4") # color is the fill colour of the bars
axes[1, 0].set_title("bar: one value per category")

# yerr is the uncertainty, fmt is the marker style, capsize is the width of the error bar caps
axes[1, 1].errorbar(groups, means, yerr=errors, fmt="o", capsize=4)
axes[1, 1].set_title("errorbar: a value plus its uncertainty")

for ax in axes.flat:            # loop over all four panels and label them
    ax.set_xlabel("x")
    ax.set_ylabel("y")

fig.suptitle("The four plot types you will use most", fontsize=13)
fig.tight_layout()              # stops the panels from overlapping each other
plt.show()

### Saving a figure

`fig.savefig` writes the figure to a file. This is how a plot leaves the notebook and ends up in
a report, a slide or a paper, and `dpi` controls how many pixels per inch it gets.

In [ ]:
# --- A finished figure, saved to a file
t = np.linspace(0, 2, 400)
slow = np.sin(2 * np.pi * 1.5 * t)
fast = 0.5 * np.sin(2 * np.pi * 5.0 * t)

fig, ax = plt.subplots()
ax.plot(t, slow, lw=2, label="1.5 Hz")
ax.plot(t, fast, lw=2, label="5 Hz, half amplitude")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude (a.u.)")
ax.set_title("Two sinusoids")
ax.set_xlim(0, 2)
ax.legend(loc="upper right", frameon=False)
fig.tight_layout()
fig.savefig("figure_two_sinusoids.png", dpi=150)   # written next to this notebook
plt.show()
print("saved figure_two_sinusoids.png")

> **▶ Task 1.2 ⭐ - Make the figure yours.**
> Copy the cell above into the empty cell below and change it so that:
> 1. the two lines are dashed rather than solid (look up the `ls` argument of `ax.plot`);
> 2. the legend sits in the lower left;
> 3. the file is saved as `my_first_figure.png`.

In [ ]:
# TODO 1.2 - the working figure is below. Change the three marked lines, then run the cell.
fig, ax = plt.subplots()
ax.plot(t, slow, lw=2, label="1.5 Hz")                  # (1) make this line dashed (see `ls`)
ax.plot(t, fast, lw=2, label="5 Hz, half amplitude")    # (1) and this one too
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude (a.u.)")
ax.set_title("Two sinusoids")
ax.set_xlim(0, 2)
ax.legend(loc="upper right", frameon=False)             # (2) move the legend to the lower left
fig.tight_layout()
fig.savefig("figure_two_sinusoids.png", dpi=150)        # (3) save it as my_first_figure.png
plt.show()

<a id="section-1-5"></a>
## 1.5 The function zoo

Five shapes cover most of what you will meet. For each one, the thing to learn is not the
formula but **what each parameter does to the picture**.

### Exponential and logarithm

The exponential $e^{ax}$ grows (or decays, if $a<0$) by a constant *factor* per unit of x. The
logarithm is its inverse: it undoes it. Because of that, plotting an exponential on a
**logarithmic y axis** turns it into a straight line, and the slope of that line is the rate.
That is the single most useful diagnostic plot in this section: if your data look straight on a
log axis, an exponential is a good model for them.

The third panel below uses **`ax.semilogy`**, which is simply `ax.plot` with the y axis already set
to a logarithmic scale. It is one call instead of two, and everything else about it behaves like a
normal plot. (`ax.semilogx` and `ax.loglog` do the same for the other axis and for both.)

In [ ]:
# --- Exponential and logarithm, on linear and on log axes
x = np.linspace(0.05, 3, 300)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))

axes[0].plot(x, np.exp(x), lw=2, label="exp(x)")
axes[0].plot(x, np.exp(-x), lw=2, label="exp(-x)")
axes[0].set_title("exponential: growth and decay") # set the title of the panel
axes[0].legend(frameon=False) # remove the box around the legend

axes[1].plot(x, np.log(x), lw=2, color="#5b2c83")
axes[1].axhline(0, color="gray", lw=0.8) # horizontal reference line at y = 0
axes[1].axvline(1, color="gray", lw=0.8, ls="--") # vertical reference line at x = 1
axes[1].set_title("logarithm: the inverse (log(1) = 0)")

axes[2].semilogy(x, np.exp(2 * x), lw=2, label="exp(2x)")
axes[2].semilogy(x, np.exp(0.5 * x), lw=2, label="exp(0.5x)")
axes[2].set_title("log y axis: exponentials become straight lines")
axes[2].legend(frameon=False)

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.tight_layout()
plt.show()

### Exponential decay

The workhorse of the course. Written with the parameters you actually care about:

$$f(t) \;=\; b + A\,e^{-t/\tau}$$

- $b$ is the **baseline**, the value the signal settles to;
- $A$ is the **amplitude**, how far above baseline it starts;
- $\tau$ (tau) is the **time constant**, in the same units as $t$. After one $\tau$ the signal
  has fallen to $1/e \approx 37\%$ of its initial displacement; after $3\tau$ it is at about 5%.

A quantity you will see in papers is the **half-life**, the time to fall halfway:
$t_{1/2} = \tau \ln 2 \approx 0.69\,\tau$.

In [ ]:
# --- Exponential decay for three time constants, with tau and the half-life marked
def exp_decay(t, amplitude=1.0, tau=1.0, baseline=0.0):
    """Exponential decay from baseline + amplitude towards baseline, with time constant tau."""
    return baseline + amplitude * np.exp(-t / tau)

t = np.linspace(0, 5, 400)

fig, ax = plt.subplots()
for tau in [0.3, 1.0, 2.0]:
    # "lw" is the line width, "label" is the name for the legend
    ax.plot(t, exp_decay(t, tau=tau), lw=2, label=f"tau = {tau} s")
    # "o" is the marker style for a filled circle, ms is the marker size
    ax.plot(tau, exp_decay(tau, tau=tau), "o", color="black", ms=5)

ax.axhline(np.exp(-1), color="gray", lw=0.8, ls="--") # horizontal reference line at y = 1/e
# add a text label to the panel, at the given coordinates, in gray
ax.text(4.2, np.exp(-1) + 0.03, "1/e = 0.37", color="gray")
ax.set_xlabel("time (s)") # set the label of the horizontal axis
ax.set_ylabel("f(t)") # set the label of the vertical axis
ax.set_title("Exponential decay: the black dots sit one tau after the start")
ax.legend(frameon=False)
# a good habit even with a single panel: it keeps things from overlapping if the figure
# is resized
fig.tight_layout()
plt.show()

print(f"half-life for tau = 1 s: {1.0 * np.log(2):.3f} s")

### The Gaussian

$$g(x) \;=\; b + A\,\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

- $\mu$ is the **centre**, where the peak sits;
- $\sigma$ is the **width**, the spread on either side;
- $A$ the **amplitude** and $b$ the **baseline**, as before.

Experimentalists usually quote the width as the **full width at half maximum** (FWHM), the
width of the bump measured halfway up. It relates to sigma by
$\text{FWHM} = 2\sqrt{2\ln 2}\,\sigma \approx 2.355\,\sigma$.

In [ ]:
# --- A Gaussian with its FWHM drawn on top
def gaussian(x, mu=0.0, sigma=1.0, amplitude=1.0, baseline=0.0):
    """Gaussian bump centred on mu with width sigma (not normalised to unit area)."""
    return baseline + amplitude * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

x = np.linspace(-4, 4, 400)
mu, sigma = 0.0, 1.0
fwhm = 2 * np.sqrt(2 * np.log(2)) * sigma

fig, ax = plt.subplots()
for s in [0.5, 1.0, 2.0]:
    ax.plot(x, gaussian(x, sigma=s), lw=2, label=f"sigma = {s}")

# add horizontal line at half maximum
ax.hlines(0.5, mu - fwhm / 2, mu + fwhm / 2, color="black", lw=2)
# add a text label to the panel, at the given coordinates, horizontally aligned to the center
ax.text(mu, 0.54, f"FWHM = {fwhm:.2f}", ha="center")
ax.set_xlabel("x")
ax.set_ylabel("g(x)")
ax.set_title("Gaussian: sigma sets the width, FWHM is how we report it")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

### The sigmoid, again

$$\sigma(x;k,x_0) = \frac{1}{1+e^{-k(x-x_0)}}$$

- $x_0$ is the **midpoint**, where the curve passes through one half. It is also the
  **inflexion point**, where the curve stops bending one way and starts bending the other.
- $k$ is the **slope** or sensitivity: the steepness at the midpoint is $k/4$.

In Week 3 this same curve becomes the **psychometric function**, with x the strength of a
stimulus and y the probability of a particular choice. Then $x_0$ is the animal's threshold and
$k$ is how sharply it discriminates.

### Sine and cosine

$$s(t) = b + A\sin(2\pi f t + \varphi)$$

- $A$ is the **amplitude**, $b$ the **baseline**;
- $f$ is the **frequency** in hertz, cycles per second, and $1/f$ is the period;
- $\varphi$ is the **phase**, a shift along the time axis.

Cosine is a sine shifted by a quarter of a cycle. Both appear the moment we look at rhythms,
which is what the EEG weeks at the end of the course are about.

In [ ]:
# --- Sigmoid slope family and sinusoid parameter family, side by side
x = np.linspace(-1, 1, 400)
t = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

for k in [2, 6, 20]:
    axes[0].plot(x, sigmoid(x, slope=k), lw=2, label=f"k = {k}")
axes[0].axhline(0.5, color="gray", lw=0.8, ls="--") # add a horizontal reference line at y = 0.5
axes[0].axvline(0.0, color="gray", lw=0.8, ls="--") # add a vertical reference line at x = 0
# set the label of the horizontal and vertical axes
axes[0].set_xlabel("x"); axes[0].set_ylabel("sigma(x)")
axes[0].set_title("sigmoid: k is the steepness at the midpoint")
axes[0].legend(frameon=False)

axes[1].plot(t, np.sin(2 * np.pi * 3 * t), lw=2, label="3 Hz")
axes[1].plot(t, np.sin(2 * np.pi * 6 * t), lw=2, label="6 Hz")
axes[1].plot(t, np.sin(2 * np.pi * 3 * t + np.pi / 2), lw=2, ls="--",
             label="3 Hz, phase +pi/2")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("s(t)")
axes[1].set_title("sinusoid: frequency and phase")
# remove the box around the legend, and make the font a bit smaller so it fits
axes[1].legend(frameon=False, fontsize=9)

fig.tight_layout()
plt.show()

> **▶ Task 1.3 ⭐ - Predict, then check.**
> In the cell below, plot the exponential decay `exp_decay` (you can copy and modify
> from the cell above) for `tau = 0.5` and `tau = 1.5` on the same axes, 
> over `t` from 0 to 4 seconds, with a legend and axis labels.
>
> Before you run it, answer in a comment: at `t = 1.5 s`, which of the two curves is higher,
> and roughly by how much? Then compute the two values and see whether you were right.

In [ ]:
# TODO 1.3 - fill in the two blanks marked ____, uncomment, and run.
# My prediction: at t = 1.5 s the higher curve is the one with tau = ...,
#                by roughly a factor of ...

# t = np.linspace(0, 4, 400)
#
# fig, ax = plt.subplots()
# for tau in [____, ____]:                    # (1) the two time constants, 0.5 s and 1.5 s
#     ax.plot(t, exp_decay(t, tau=tau), lw=2, label=f"tau = {tau} s")
# ax.set_xlabel("time (s)")
# ax.set_ylabel("f(t)")
# ax.set_title("Two decay rates")
# ax.legend(frameon=False)
# fig.tight_layout()
# plt.show()
#
# (2) now check your prediction against the numbers
# print("at t = 1.5 s:", exp_decay(1.5, tau=____), exp_decay(1.5, tau=____))

### One last exercise, which pulls Part 1 together

That is the zoo. Before moving on to calculus, here is an exercise that uses three of the things
you have met in this part at once: you will take one of these shapes, write it into **your own
module** the way section 1.2 did with the sigmoid, import it back, and **plot** it.

One technical detail first, because it will trip you up the moment you edit the file.

> **Iterating on the module.** If you edit `src/qmn_utils.py` while the kernel is running,
> Python keeps using the version it imported first, and your edits stay invisible until you
> restart the kernel. To pick them up without restarting:
>
> ```python
> import importlib, src.qmn_utils
> importlib.reload(src.qmn_utils)
> from src.qmn_utils import gaussian    # re-bind the name in the notebook
> ```

> **▶ Task 1.4 ⭐⭐ - Move a function into the module.**
> Section 1.2 introduced `src/qmn_utils.py` and imported `sigmoid` out of it. Now put
> something of your own in there.
>
> Add the `gaussian(x, mu, sigma, amplitude, baseline)` function from section 1.5 to
> `src/qmn_utils.py` (open it in VS Code, paste it in, save). Then reload the module with the
> trick above, import `gaussian` from it, and plot it for `mu = 0.2` and `sigma = 0.1`.
> From now on, anything you put in `qmn_utils.py` is available in every notebook of the course.

In [ ]:
# TODO 1.4
# (1) Open src/qmn_utils.py in VS Code, paste the gaussian() function from section 1.5 into
#     it, and save the file. That is the only part you have to write yourself.
# (2) Then uncomment everything below and run this cell. The plot is already written for you,
#     so it will work as soon as the import succeeds.

# import importlib, src.qmn_utils
# importlib.reload(src.qmn_utils)
# from src.qmn_utils import gaussian as gaussian_from_module
#
# x = np.linspace(-0.5, 1.0, 300)
# fig, ax = plt.subplots()
# ax.plot(x, gaussian_from_module(x, mu=0.2, sigma=0.1), lw=2, color="#5b2c83")
# ax.set_xlabel("x")
# ax.set_ylabel("g(x)")
# ax.set_title("Gaussian, imported from src/qmn_utils.py")
# fig.tight_layout()
# plt.show()

<a id="section-2"></a>
# Part 2. Calculus, numerically

In the lecture, derivatives and integrals were defined as limits. On a computer we never take a
limit: we have a function sampled at finitely many points, and we approximate. This part builds
those approximations and checks them against the analytic answers, which is the honest way to
find out how far you can trust them.

<a id="section-2-1"></a>
## 2.1 The derivative as a local slope

The derivative $f'(x)$ is the slope of $f$ at $x$: how much the output changes per unit of
input. Numerically, the slope between two neighbouring samples is

$$f'(x) \;\approx\; \frac{f(x+h) - f(x-h)}{2h}$$

which is what `np.gradient(y, x)` computes for every point of an array at once.

We test it on the Gaussian, whose analytic derivative we can write down:

$$g(x) = e^{-x^2/2\sigma^2} \quad\Longrightarrow\quad g'(x) = -\frac{x}{\sigma^2}\,g(x)$$

In [ ]:
# --- Numerical derivative vs the analytic one
x = np.linspace(-4, 4, 400)
g = gaussian(x, mu=0.0, sigma=1.0)

dg_numeric = np.gradient(g, x)          # slope at every point, from the samples
dg_analytic = -x * g                    # with sigma = 1

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(x, g, lw=2)
axes[0].set_title("g(x), a Gaussian")
axes[1].plot(x, dg_numeric, lw=3, alpha=0.5, label="np.gradient")
axes[1].plot(x, dg_analytic, lw=1.5, ls="--", color="black", label="analytic")
axes[1].axhline(0, color="gray", lw=0.8)
axes[1].set_title("g'(x): the two agree")
axes[1].legend(frameon=False)
for ax in axes:
    ax.set_xlabel("x")
fig.tight_layout()
plt.show()

print(f"largest disagreement: {np.abs(dg_numeric - dg_analytic).max():.2e}")

Notice where the derivative is zero: exactly at the peak of the Gaussian. That is the general
rule, a maximum or a minimum of $f$ is a **zero of $f'$**, and it is how optimisation
algorithms find peaks. Notice also that the derivative is largest in magnitude where the curve
is steepest, which is roughly one sigma either side of the centre.

The quality of the approximation depends on how finely you sample. With 400 points over 8 units
the agreement above is excellent. Try changing 400 to 40 and look at the number printed.

<a id="section-2-2"></a>
## 2.2 The second derivative: curvature

Differentiate twice and you get the **curvature**. The two signs tell you two different things:

- the sign of $f'$ says whether the function is **increasing** or **decreasing**;
- the sign of $f''$ says whether it is **convex** (curving upwards, like a valley) or
  **concave** (curving downwards, like a hill).

Where $f''$ crosses zero the curve changes its bending: that is an **inflexion point**. For the
sigmoid, that is exactly the midpoint, which is why the midpoint is also the steepest point.

In [ ]:
# --- A sigmoid, its slope and its curvature, stacked
x = np.linspace(-1, 1, 500)
s = sigmoid(x, slope=8.0)
ds = np.gradient(s, x)
d2s = np.gradient(ds, x)

fig, axes = plt.subplots(3, 1, figsize=(7, 7), sharex=True)
for ax, y, name in zip(axes, [s, ds, d2s],
                       ["f(x): the sigmoid", "f'(x): slope, peaks at the midpoint",
                        "f''(x): curvature, crosses zero at the midpoint"]):
    ax.plot(x, y, lw=2)
    ax.axvline(0, color="gray", lw=0.8, ls="--")
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_ylabel(name.split(":")[0])
    ax.set_title(name, fontsize=10)
axes[-1].set_xlabel("x")
fig.tight_layout()
plt.show()

print(f"steepest slope, measured : {ds.max():.2f}")
print(f"steepest slope, predicted (k/4): {8.0 / 4:.2f}")

<a id="section-2-3"></a>
## 2.3 The integral as accumulation

If the derivative is a rate, the integral is a **total**. The integral of $f$ between $a$ and
$b$ is the area under the curve, and the way to compute it numerically is to chop that area
into thin strips and add them up. `np.trapezoid` does that with trapezoidal strips.

Two different questions, two different tools:

- "what is the total?" is one number: `np.trapezoid(y, x)`;
- "how does the total build up along the way?" is a whole curve, the **cumulative** integral:
  `np.cumsum(y) * dx`.

We check the first against a case where we know the answer: a Gaussian with amplitude 1 and
width sigma has area $\sigma\sqrt{2\pi}$.

In [ ]:
# --- Total area, and how it accumulates
# np.cumsum(...) * dt is the running sum of rectangles. Two neighbours worth knowing: np.trapz
# does the same total with trapezoids instead of rectangles, and np.diff is the raw difference
# between neighbouring samples, which is what np.gradient smooths over.
x = np.linspace(-6, 6, 1000)
dx = x[1] - x[0]
sigma = 1.0
g = gaussian(x, mu=0.0, sigma=sigma)

area_numeric = np.trapezoid(g, x) # the area under the curve, from the samples
area_analytic = sigma * np.sqrt(2 * np.pi) # the area under a Gaussian is known analytically
running_area = np.cumsum(g) * dx # the area accumulated up to each x

print(f"numeric area : {area_numeric:.6f}")
print(f"analytic area: {area_analytic:.6f}")
print(f"running area at several points (x,y pairs):")
for xi, yi in zip(x[::200], running_area[::200]):
    print(f"  ({xi:.2f}, {yi:.6f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(x, g, lw=2)
axes[0].fill_between(x, g, alpha=0.25)
axes[0].set_title(f"the area under this curve is {area_numeric:.3f}")
axes[1].plot(x, running_area, lw=2, color="#5b2c83")
axes[1].axhline(area_analytic, color="gray", lw=0.8, ls="--")
axes[1].set_title("the same area, accumulating from left to right")
for ax in axes:
    ax.set_xlabel("x")
fig.tight_layout()
plt.show()

<a id="section-2-4"></a>
## 2.4 The Fundamental Theorem of Calculus 🔬

> 🔬 **Optional.** The formal statement behind the two sections above. You do not need it to use a
> derivative or an integral in the weeks that follow.

The theorem says differentiation and integration undo each other. Integrate a function to get
its cumulative curve $F$, then differentiate $F$, and you are back to the function you started
from:

$$\frac{d}{dx}\int_a^x f(u)\,du \;=\; f(x)$$

This is not an abstraction: it is the statement that the *slope of the accumulated total* is
the *rate at which it is accumulating*. Speed and distance are the everyday version. Below we
check it numerically, which takes three lines.

In [ ]:
# --- Integrate, then differentiate, and compare with the original
x = np.linspace(-4, 4, 800)
dx = x[1] - x[0]

f = gaussian(x, mu=0.0, sigma=1.0)
# get a running integral of f(x) from the left, using the trapezoid rule
F = np.cumsum(f) * dx
back = np.gradient(F, x)         # then differentiate

fig, ax = plt.subplots()
ax.plot(x, f, lw=3, alpha=0.5, label="f(x), the original")
ax.plot(x, back, lw=1.5, ls="--", color="black", label="d/dx of the running integral")
ax.set_xlabel("x"); ax.set_ylabel("value")
ax.set_title("The Fundamental Theorem of Calculus, numerically")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print(f"largest disagreement: {np.abs(f - back).max():.2e}")

> **▶ Task 2.1 ⭐⭐ - Verify the theorem on a different function.**
> Repeat the check above for the **exponential decay** `exp_decay(t, amplitude=2, tau=0.8)` on
> `t` from 0 to 5 seconds with 600 points. Complete the three missing lines below, plot the
> original against the differentiated running integral, and print the largest disagreement.
>
> Then answer in a comment: the disagreement is largest at one end of the interval. Which end,
> and why? (Hint: think about what `np.gradient` can do at the very first and very last point,
> where it has a neighbour on only one side.)

In [ ]:
# TODO 2.1 - fill in the three blanks marked ____, uncomment, and run.
# t = np.linspace(0, 5, 600)
# dt = t[1] - t[0]
#
# f = ____                     # (1) exp_decay evaluated on t, with amplitude 2 and tau 0.8
# F = ____                     # (2) accumulate it:  np.cumsum(...) * dt
# back = ____                  # (3) differentiate the accumulation:  np.gradient(..., t)
#
# --- the figure and the check are written for you
# fig, ax = plt.subplots()
# ax.plot(t, f, lw=3, alpha=0.5, label="f(t)")
# ax.plot(t, back, lw=1.5, ls="--", color="black", label="d/dt of the running integral")
# ax.set_xlabel("time (s)"); ax.set_ylabel("value")
# ax.set_title("FTC on an exponential decay")
# ax.legend(frameon=False)
# fig.tight_layout()
# plt.show()
# print(f"largest disagreement: {np.abs(f - back).max():.2e}")

# Then answer here: at which end of the interval is the disagreement largest, and why?

<a id="section-2-5"></a>
## 2.5 Local linearization: Taylor to first order 🔬

> 🔬 **Optional.** One of the most useful ideas in applied maths, and nothing in this course depends
> on it. Read it the day you meet a model that needs it.

Near a point $x_0$, any smooth function looks like a straight line. The **first-order Taylor
approximation** is that line:

$$f(x) \;\approx\; f(x_0) + f'(x_0)\,(x - x_0)$$

It is the tangent at $x_0$. This is the idea behind a great deal of applied maths: replace an
awkward function by a line, valid as long as you do not stray too far. The interesting question
is always *how far is too far*, and the answer is visible in a plot.

In [ ]:
# --- A sigmoid and its tangent at the midpoint, plus the error
x = np.linspace(-1, 1, 500)
s = sigmoid(x, slope=8.0)
ds = np.gradient(s, x)

x0 = 0.0
i0 = np.argmin(np.abs(x - x0))                       # index of the point nearest x0
tangent = s[i0] + ds[i0] * (x - x0)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(x, s, lw=2, label="sigmoid")
axes[0].plot(x, tangent, lw=1.5, ls="--", color="black", label="tangent at x = 0")
axes[0].plot(x0, s[i0], "o", color="red", ms=6)
axes[0].set_ylim(-0.1, 1.1)
axes[0].set_title("the line is a good stand-in near x = 0")
axes[0].legend(frameon=False)

axes[1].plot(x, np.abs(s - tangent), lw=2, color="#d62728")
axes[1].set_title("absolute error of the approximation")
for ax in axes:
    ax.set_xlabel("x")
fig.tight_layout()
plt.show()

for width in [0.05, 0.1, 0.3]:
    inside = np.abs(x - x0) < width
    print(f"within +/- {width}: largest error {np.abs(s - tangent)[inside].max():.4f}")

> **▶ Task 2.2 ⭐⭐ - Where does the line stop working?**
> Using the printout above as a model, find (roughly) how far from `x = 0` you can go before
> the tangent is wrong by more than 0.02, for a sigmoid with `slope = 8`. Then repeat for
> `slope = 20`. Does a steeper sigmoid give you a wider or a narrower region where the linear
> approximation is good? Explain the answer in one sentence.

In [ ]:
# TODO 2.2 - fill in the two blanks marked ____, uncomment, and run.
# x = np.linspace(-1, 1, 2001)
#
# for k in [8.0, 20.0]:
#     s = sigmoid(x, slope=k)
#     ds = np.gradient(s, x)
#     i0 = np.argmin(np.abs(x))          # index of the point at x = 0
#     tangent = s[i0] + ds[i0] * x
#
#     error = ____                       # (1) absolute difference between s and tangent
#     good = ____                        # (2) the largest |x| where error stays below 0.02
#                                        #     hint: np.abs(x)[error <= 0.02].max()
#     print(f"slope k = {k:4.1f}: the tangent stays within 0.02 up to about +/- {good:.3f}")

# Then answer here: does a steeper sigmoid give a wider or a narrower good region, and why?

<a id="section-3"></a>
# Part 3. Putting it together on generated signals 🔬

> 🔬 **Optional.** Two miniature analyses that use everything above. Nothing later in the
> course depends on them, and they are the natural place to stop if the session runs out.

Both are the shape of something you will do for real later on: measure a parameter from a
noisy trace, and find the moments a signal changes.

<a id="section-3-1"></a>
## 3.1 A calcium transient

When a neuron fires, a calcium indicator such as GCaMP brightens quickly and then decays back
to baseline roughly exponentially. The decay time constant is a property of the indicator and
of the cell, and reading it off a trace is a standard measurement.

We build a transient as a fast rise multiplied by a slow decay, add a little measurement noise,
and then recover tau from the noisy trace.

In [ ]:
# --- Build a noisy calcium transient
rng = np.random.default_rng(2)

t = np.linspace(0, 6, 600)          # six seconds, 600 samples
t_onset, tau_rise, tau_decay = 1.0, 0.08, 1.2   # the spike happens at 1 s; rise fast, decay slow
amplitude, baseline = 1.0, 0.05                 # how far it climbs, and where it sits at rest

# A mask: True for every sample after the spike, False before it. Nothing happens before onset.
after = t >= t_onset

# Start from a flat trace sitting at the baseline everywhere, the same shape as t.
clean = np.full_like(t, baseline)

# After the onset the trace is the product of two exponentials, which is the standard shape of a
# calcium transient: one term climbing from 0 to 1 with the fast time constant tau_rise, and one
# falling back towards 0 with the slow tau_decay. Fast in, slow out.
rise = 1 - np.exp(-(t[after] - t_onset) / tau_rise)
decay = np.exp(-(t[after] - t_onset) / tau_decay)
clean[after] = baseline + amplitude * rise * decay

# Real recordings are noisy, so add a little Gaussian measurement noise on top.
noisy = clean + rng.normal(0, 0.03, size=t.size)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(t, noisy, lw=0.8, color="0.6", label="measured (with noise)")
ax.plot(t, clean, lw=2, label="underlying signal")
ax.axvline(t_onset, color="gray", lw=0.8, ls="--")
ax.set_xlabel("time (s)"); ax.set_ylabel("dF/F")
ax.set_title("A simulated calcium transient")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

To measure the decay we use the fact from section 1.5: after one time constant an exponential has
fallen to $1/e$, about 37%, of its starting height. So tau can be read straight off the trace. Find
the peak, work out the level one time constant below it, and measure how long the trace takes to
get there.

There is a better way, which uses every sample of the decay instead of the one that happens to
cross the line, and it is called fitting. That is Week 7's subject; one crossing is enough here.

In [ ]:
# --- Read the decay time constant off the trace, with no fitting
peak_index = np.argmax(noisy)
peak_time, peak_value = t[peak_index], noisy[peak_index]

# the level one time constant below the peak: 1/e of the way down towards the baseline
target = baseline + (peak_value - baseline) / np.e      # np.e is Euler's number, 2.718...

def tau_from_crossing(trace):
    """Time from the peak of a trace to the moment it has fallen to 1/e of its height."""
    peak = np.argmax(trace)
    level = baseline + (trace[peak] - baseline) / np.e
    after_peak = np.arange(t.size) > peak            # a mask: samples after the peak only
    below = after_peak & (trace < level)             # ... that have already fallen far enough
    return t[np.argmax(below)] - t[peak]             # argmax on True/False finds the first True

tau_noisy = tau_from_crossing(noisy)
tau_clean = tau_from_crossing(clean)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(t, noisy, lw=0.8, color="0.6", label="measured")
ax.plot(t, clean, lw=2, label="underlying signal")
ax.axhline(target, color="#d62728", lw=1.2, ls="--", label="one time constant below the peak")
ax.plot([peak_time, peak_time + tau_noisy], [target, target], color="#d62728", lw=4)
ax.set_xlabel("time (s)"); ax.set_ylabel("dF/F")
ax.set_title("Reading tau off the trace: the red bar is the estimate")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print(f"true tau                  : {tau_decay:.2f} s")
print(f"estimated from the clean trace : {tau_clean:.2f} s")
print(f"estimated from the noisy trace : {tau_noisy:.2f} s")
print(f"half-life of the estimate      : {tau_noisy * np.log(2):.2f} s")

Neither number lands on 1.2 s, and they miss in opposite directions, for two different reasons.

On the **clean** trace the rule reads a little long. The peak is not the start of the decay: the
rising term is still lifting the trace when the peak occurs, so the fall to $1/e$ takes slightly
longer than one time constant.

On the **noisy** trace it reads short. The measurement ends at the *first* sample below the line,
and noise gives an early sample plenty of chances to dip under it. One unlucky sample is enough.

Both are real measurement problems rather than coding mistakes, and both get better if you use the
whole decay instead of a single crossing. That is what fitting does, in Week 7.

> **▶ Task 3.1 ⭐⭐ - The rate of change of the transient.**
> Compute the numerical derivative of the **clean** transient with `np.gradient` and plot it
> under the signal, sharing the x axis. Then answer in comments:
> 1. at what time is the derivative largest, and what is happening to the signal there?
> 2. where does the derivative cross zero, and what does that instant correspond to?

In [ ]:
# TODO 3.1 - fill in the three blanks marked ____, uncomment, and run.
# rate = ____                    # (1) numerical derivative of `clean` with respect to t
#
# --- the figure is written for you
# fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
# axes[0].plot(t, clean, lw=2)
# axes[0].set_ylabel("dF/F"); axes[0].set_title("signal")
# axes[1].plot(t, rate, lw=2, color="#d62728")
# axes[1].axhline(0, color="gray", lw=0.8)
# axes[1].set_ylabel("d(dF/F)/dt"); axes[1].set_xlabel("time (s)")
# axes[1].set_title("its rate of change")
# fig.tight_layout()
# plt.show()
#
# print("the derivative is largest at t =", ____)    # (2) hint: t[np.argmax(...)]
# print("the signal peaks at        t =", ____)      # (3) the same idea, on the signal itself

# Then answer here: 1. what is happening to the signal where the derivative is largest?
#                   2. what does the instant where the derivative crosses zero correspond to?

<a id="section-3-2"></a>
## 3.2 When did the animal learn?

An animal learning a psychophysics task of Week 3 does not necessarily improve smoothly over time.
It can stay for a while at chance level, then improves quickly when it works out one part of the task, settles at a new level, and later improves again when it works out the next part. Plotted against session number, the performance looks like a staircase rather than a ramp.

Each step of that staircase is a **sigmoid**, the shape from section 1.5: flat, then a rapid rise,
then flat again at a higher level. A learning curve with three steps is the sum of three sigmoids
on top of a baseline, and the baseline here is 0.5, because a mouse guessing between two sides is
right half the time.

Two questions get asked of a curve like this in practice, and both are answered with a derivative
and a comparison:

- **When did each transition happen?** That is where the curve is climbing fastest, so it is where
  the derivative has a peak.
- **When did the animal reach criterion?** That is the first session where performance passes a
  threshold, 0.8 in the IBL protocol, which is how an animal is declared trained.

In [ ]:
# --- A learning curve built out of three transitions
def transition(session, midpoint, height, sharpness):
    """One learning step: a sigmoid climbing by `height`, centred on session `midpoint`."""
    return height / (1 + np.exp(-sharpness * (session - midpoint)))

sessions = np.arange(0, 61.0)     # one value per session, sixty one of them

CHANCE = 0.5                      # two sides to choose from, so guessing is right half the time
steps = [                         # midpoint, height, sharpness
    (12, 0.15, 0.8),              # starts responding to the stimulus at all
    (28, 0.18, 0.5),              # works out that the side of the grating is what matters
    (45, 0.07, 1.2),              # starts getting the low contrasts right too
]

learning = np.full_like(sessions, CHANCE)
for midpoint, height, sharpness in steps:
    learning = learning + transition(sessions, midpoint, height, sharpness)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(sessions, learning, "o-", lw=2, ms=4)
ax.axhline(CHANCE, color="gray", lw=0.8, ls=":", label="chance")
ax.set_xlabel("session")
ax.set_ylabel("proportion correct")
ax.set_title("Three transitions, one staircase")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
plt.show()

print(f"the curve runs from {learning[0]:.2f} to {learning[-1]:.2f}")

### Finding the first point that satisfies a condition

Comparing an array with a number gives an array of `True` and `False`, one per sample. Feeding that
to `np.argmax` gives the position of the first `True`, because `True` counts as 1 and `False` as 0,
and `argmax` stops at the first largest value. It is the same trick section 3.1 used to find where
the transient had fallen to $1/e$.

There is a trap in it worth knowing now rather than later. If **no** sample satisfies the
condition, the array is all `False`, and `np.argmax` returns 0: not an error, not a warning, just
the first position, which looks like a perfectly good answer. So always ask whether the condition
happens at all before believing where it happened.

In [ ]:
# --- The first session at criterion, and the trap that comes with it
CRITERION = 0.8

reached = learning >= CRITERION            # an array of True and False, one per session
if reached.any():                          # ask first: does it ever happen?
    first = np.argmax(reached)             # the position of the first True
    print(f"criterion of {CRITERION} first reached at session {sessions[first]:.0f}, "
          f"where the curve reads {learning[first]:.3f}")
else:
    print("this animal never reaches criterion")

# The trap, made visible: nothing here is above 2.0, and argmax answers anyway.
impossible = learning >= 2.0
print(f"\nany session above 2.0? {impossible.any()}")
print(f"np.argmax says the first one is session {sessions[np.argmax(impossible)]:.0f}, "
      f"which is simply the start of the array")

### Finding the peaks of the derivative

The transitions are where the curve climbs fastest, so they are the peaks of its derivative. A peak
is a sample larger than its neighbours, and `scipy.signal.find_peaks` finds them for you. Two of its
arguments are worth knowing: `height` ignores bumps below a given size, and `distance` refuses to
report two peaks closer together than you say. Both are choices you make, not facts the data hands
you.

In [ ]:
# --- The transitions, as peaks of the rate of improvement
from scipy.signal import find_peaks

rate = np.gradient(learning, sessions)               # improvement per session
peaks, properties = find_peaks(rate, height=0.005)   # ignore anything smaller than 0.005

fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
axes[0].plot(sessions, learning, "o-", lw=2, ms=4)
axes[0].axhline(CRITERION, color="#072A92", lw=1.2, ls="--", label=f"criterion {CRITERION}")
axes[0].plot(sessions[first], learning[first], "o", ms=10, color="#072A92")
axes[0].set_ylabel("proportion correct")
axes[0].set_title("the learning curve, with the session it reaches criterion")
axes[0].legend(frameon=False, loc="lower right")

axes[1].plot(sessions, rate, lw=2, color="#d62728")
axes[1].plot(sessions[peaks], rate[peaks], "v", ms=10, color="black", label="transitions")
axes[1].axhline(0, color="gray", lw=0.8)
axes[1].set_xlabel("session")
axes[1].set_ylabel("d(correct)/d(session)")
axes[1].set_title("its derivative: one peak per transition")
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()

print(f"{len(peaks)} transitions found, at sessions "
      f"{', '.join(f'{s:.0f}' for s in sessions[peaks])}")
print(f"the animal put in for real: {', '.join(str(m) for m, _, _ in steps)}")

The derivative lands on the three midpoints exactly, and it says *when* the animal changed, which
the accuracy curve does not show with any precision.

Read the heights carefully, though. A peak is tall when a transition is large **or** when it is
fast, so the two are confounded in it: the second step is the largest of the three, 0.18, and its
peak is still the lower, because the first step is worth less but happens faster. What separates
size from speed is the area under each peak, which is exactly the height gained in that step. That
is section 2.3 again, the integral as accumulation, doing something useful.

Notice also where criterion falls. Session 32 is where the second transition has just finished and
the third has not started, so an animal declared trained there still had the whole of its last
transition ahead of it. A criterion is a convention, not the end of learning.

One honest caveat. This curve was built rather than measured, so it is perfectly smooth. A real
learning curve wobbles from session to session, and differentiating a wobbly curve amplifies the
wobble, because a derivative responds to the fastest changes in a signal and noise is the fastest
thing in it. Getting a usable derivative out of real data therefore takes a step we are not doing
here, and Week 12 is where it happens.

> **▶ Task 3.2 ⭐⭐⭐ - A second animal, from start to finish.**
> A second mouse learns the same task, more slowly. Its three transitions are given in the cell
> below and everything else is yours to write, using the two cells above as your model.
>
> 1. Build its learning curve from `slow_steps`, the same way as above.
> 2. Find the first session at which it reaches `CRITERION` and print it. Make sure it gets there
>    at all before you believe the answer.
> 3. Take its derivative and find the transitions with `find_peaks`. Print the sessions they fall
>    on next to the ones that went in.
> 4. Draw a figure with **two panels sharing the x axis**. On top, both learning curves, the first
>    animal and the second, with the criterion drawn as a horizontal line. Underneath, the second
>    animal's derivative, with a marker on each peak that was detected.
>
> Then answer in a comment: the two animals reach criterion within a few sessions of each other.
> What does the derivative show that the two accuracy curves hide?

In [ ]:
# TODO 3.2 - the second animal is described here, the analysis and the figure are yours.
slow_steps = [(18, 0.22, 0.35), (34, 0.10, 0.6), (50, 0.06, 0.9)]   # midpoint, height, sharpness

# 1. build the curve
# 2. the first session at criterion
# 3. the derivative, and its peaks
# 4. the two panel figure

# Then answer here: what does the derivative show that the two accuracy curves hide?

<a id="section-4"></a>
# 4. What comes next

You now have the two habits the rest of the course rests on: build the object, then plot it and
look at it. Specifically, you can write a vectorized function, dress a figure properly, and
compute a derivative or an integral without an analytic formula.

Three threads from today continue directly:

- The **sigmoid** returns in Week 3 as the psychometric curve of a mouse making decisions,
  with its midpoint becoming a perceptual threshold.
- The **exponential decay** returns whenever something relaxes back to a baseline, and the
  $1/e$ rule from section 3.1 is the quickest way to measure one.
- The **staircase** of section 3.2 is next week's subject seen from the other side: Week 3
  opens the real learning curves of thirty mice, and the criterion we crossed by hand there is
  the one that splits the dataset into its untrained and trained phases.
- **Sines and cosines** come back at the very end, in the Fourier weeks, where a real
  recording is taken apart into the rhythms that compose it.

Next week we stop generating our own data and open a real one: the behaviour of 30 mice
learning a visual decision task.